# 久遠真和 Sin 的自我介绍 · MiniMax H3 生成笔记本

在租来的显卡上用 **MiniMax H3（开源权重，Ref2VA 参考生成）** 生成分镜里的七个镜头。镜头、提示词、参考图分组都在 `shots.json`，这个笔记本只负责跑。

**先看许可证**：H3 开源权重目前不允许在美国、欧盟、英国、韩国部署，要用先在 <https://platform.minimax.io/h3-license> 申请（见官方 `docs/QA-about-License.md`）。Colab 的机器在美国。笔记本在 Colab 和普通 Jupyter（国内算力平台）上都能跑。

**显卡**：至少 44GB 显存，推荐 80GB（A100 80GB / H100 / H800 / RTX PRO 6000）。

**Drive（或工作目录）里要先放好：**

```
shin-intro/
  shots.json            分镜（仓库 scripts/intro-video/shots.json）
  refs/                 参考图和声音
    ref1-shin-full.png  ref2-shin-face.png  ref3-shin-turnaround.png
    ref4-shin-coding.png  ref5-sin-crayon.png
    sin-voice.wav       你自己录的 5–10 秒人声（安静的房间，随便说一段话）
  workflows/            第 6 步在 ComfyUI 里导出的 API 工作流（第一次运行时还没有）
```

按顺序运行。第 1–5 步每次开机都要跑（机器关掉后环境就没了），第 6 步只做一次。

## 1. 环境和显卡

In [ ]:
import json, os, shutil, subprocess, sys, time, pathlib, zlib

IN_COLAB = "google.colab" in sys.modules
smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
if smi.returncode != 0:
    raise SystemExit("没有检测到 NVIDIA 显卡：Colab 里在“代码执行程序 → 更改运行时类型”选 A100 / H100 / G4")
GPU_NAME, GPU_MB = [x.strip() for x in smi.stdout.strip().splitlines()[0].split(",")]
GPU_MB = int(GPU_MB)
print(f"显卡：{GPU_NAME}，显存 {GPU_MB / 1024:.0f} GB")
if GPU_MB < 44_000:
    raise SystemExit("显存不到 44GB，跑不动 H3。换 A100 80GB / H100 / G4")
if GPU_MB < 70_000:
    print("⚠️ 显存不到 80GB：先用 4 秒的短片段试，爆显存就换更大的卡")

# 按显卡选权重精度：H100/H800 原生支持 fp8；Blackwell（RTX PRO 6000 / B 系列）支持 nvfp4；其他（A100 等）用 int8
blackwell = any(k in GPU_NAME for k in ("RTX PRO 6000", "B200", "B100", "Blackwell", "RTX 50"))
hopper = any(k in GPU_NAME for k in ("H100", "H800", "H200", "H20"))
DIT = "minimax_h3_ref2va_pruned_fp8_scaled.safetensors" if (hopper or blackwell) else "minimax_h3_ref2va_pruned_int8_convrot.safetensors"
TEXT_ENCODER = "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors" if blackwell else "qwen3vl_32b_minimax_h3_int8_convrot.safetensors"
MODEL_FILES = [
    f"diffusion_models/{DIT}",
    f"text_encoders/{TEXT_ENCODER}",
    "vae/minimax_h3_video_vae_fp16.safetensors",
    "vae/minimax_h3_audio_vae_fp32.safetensors",
]
DRAFT_LORA = True   # 4 步出图的加速 LoRA，打草稿用（约 2GB）
if DRAFT_LORA:
    MODEL_FILES.append("loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors")
EXTRA_FILES = []    # ComfyUI 模板弹窗如果要的文件不在上面，把“文件夹/文件名”加到这里再运行第 4 步
print("要用的模型文件：", *MODEL_FILES + EXTRA_FILES, sep="\n  ")

## 2. 工作目录

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = pathlib.Path("/content/drive/MyDrive/shin-intro")
    BASE = pathlib.Path("/content")
else:
    WORK = pathlib.Path(os.environ.get("SHIN_INTRO_DIR", "shin-intro")).resolve()
    BASE = pathlib.Path(os.environ.get("COMFY_BASE", ".")).resolve()

data = json.loads((WORK / "shots.json").read_text())
GROUPS, SHOTS = data["groups"], data["shots"]
need = sorted({f for g in GROUPS.values() for f in g["images"] + g["audio"]})
missing = [f for f in need if not (WORK / "refs" / f).exists()]
print("工作目录：", WORK)
print("镜头：", ", ".join(f'{s["id"]}({s["seconds"]}s)' for s in SHOTS))
print("还缺的参考文件：", missing or "无")
if "sin-voice.wav" in missing:
    print("⚠️ 先录一段你自己的声音 sin-voice.wav 放进 refs/（镜头“本子里的 Sin”要用）")
if "shin-voice.wav" in missing:
    print("ℹ️ shin-voice.wav 在第 8 步从“名片”里截出来，现在缺是正常的")
for d in ("workflows", "takes"):
    (WORK / d).mkdir(parents=True, exist_ok=True)

## 3. 安装 ComfyUI

In [ ]:
COMFY = BASE / "ComfyUI"
if not COMFY.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI", str(COMFY)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(COMFY / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf_transfer"], check=True)
if shutil.which("ffmpeg") is None:
    subprocess.run(["bash", "-c", "apt-get -qq update && apt-get -qq install -y ffmpeg"], check=True)
print("ComfyUI 就绪：", subprocess.run(["git", "-C", str(COMFY), "log", "-1", "--format=%h %cd"], capture_output=True, text=True).stdout.strip())

## 4. 下载模型（约 55GB，每次开机都要下）

国内机器把 `HF_ENDPOINT` 改成镜像地址（例如 `https://hf-mirror.com`），或者从 ModelScope 的 MiniMax-H3 镜像下载后放进对应文件夹。

In [ ]:
HF_ENDPOINT = ""   # 国内机器可填镜像地址
if HF_ENDPOINT:
    os.environ["HF_ENDPOINT"] = HF_ENDPOINT
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import hf_hub_download

start = time.time()
for f in MODEL_FILES + EXTRA_FILES:
    target = COMFY / "models" / f
    if target.exists():
        print("已存在", f); continue
    print("下载", f, "…")
    hf_hub_download(repo_id="Comfy-Org/MiniMax-H3", filename=f, local_dir=str(COMFY / "models"))
print(f"完成，用时 {(time.time() - start) / 60:.1f} 分钟")
print(subprocess.run(["df", "-h", str(COMFY)], capture_output=True, text=True).stdout)

## 5. 启动 ComfyUI，放好参考素材

In [ ]:
import urllib.request

def copy_refs():
    for f in (WORK / "refs").iterdir():
        if f.is_file():
            shutil.copy2(f, COMFY / "input" / f.name)
copy_refs()

log = open(BASE / "comfyui.log", "w")
COMFY_PROC = subprocess.Popen([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188"], cwd=COMFY, stdout=log, stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2); break
    except Exception:
        time.sleep(2)
else:
    raise SystemExit("ComfyUI 没起来，看 comfyui.log")
print("ComfyUI 已启动")
if IN_COLAB:
    from google.colab import output
    output.serve_kernel_port_as_window(8188)   # 通过 Colab 自己的代理打开，只有你登录的账号能访问
else:
    print("在平台的“自定义服务 / 端口映射”里打开 8188 端口")

如果上面的链接打开后界面连不上（一直转圈），用这一格开一条 Cloudflare 临时隧道。**这个地址任何人拿到都能用你的显卡，别发给别人，用完关掉笔记本。**

In [ ]:
# 可选：Cloudflare 临时隧道
if not pathlib.Path("/usr/local/bin/cloudflared").exists():
    subprocess.run(["bash", "-c", "curl -sL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared"], check=True)
TUNNEL = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in TUNNEL.stdout:
    if "trycloudflare.com" in line:
        print(line.strip()); break

## 6. 在 ComfyUI 里做四个工作流（只做一次）

每个“参考组”一个工作流，参考图和声音的**顺序就是提示词里的编号**（第一张图是 `<Picture 1>`，第一段声音是 `<Audio 1>`）：

| 工作流文件 | 参考图（按顺序） | 声音（按顺序） | 用在 |
|---|---|---|---|
| `shin-novoice.json` | ref1-shin-full, ref3-shin-turnaround, ref2-shin-face | 无 | 名片（定声音） |
| `shin.json` | ref1-shin-full, ref3-shin-turnaround, ref2-shin-face | shin-voice | 影子、Yozora、道别 |
| `coding.json` | ref4-shin-coding, ref2-shin-face | shin-voice | 写代码 |
| `notebook.json` | ref1-shin-full, ref5-sin-crayon, ref2-shin-face | sin-voice, shin-voice | 小本子、本子里的 Sin |

步骤：

1. 左侧“模板”→ Video → **MiniMax H3 Reference to Video**。弹出缺模型时不用下载，在加载节点里选第 1 步列出的文件（int8 / fp8 版本）。
2. 按上表接好参考图和声音，画幅选 16:9，分辨率用默认（短边 768）。
3. 提示词框里只填 `{{PROMPT}}`。如果有“AI 改写提示词”的开关，关掉（`shots.json` 里已经是改写后的格式）。
4. 打草稿：如果模板里有 LoRA 加载节点，选 `ref2v_turbo_4step` 并按它的说明把步数改成 4；出成片时把它停用。
5. 先手动点一次运行，确认能出片、记下用时和显存。
6. 菜单 → 导出（API）→ 按上表命名，上传到 `shin-intro/workflows/`。

`shin.json` 要等第 8 步截出 `shin-voice.wav` 之后再做（在 `shin-novoice` 上加一个声音节点）。

## 7. 生成工具（运行一次即可）

In [ ]:
import copy, uuid, urllib.parse
from IPython.display import Video, display

API = "http://127.0.0.1:8188"
CLIENT = str(uuid.uuid4())
SEED_KEYS = {"seed", "noise_seed"}
DURATION_KEYS = {"duration", "seconds", "length_seconds", "video_duration"}

def patch_workflow(wf, prompt, seed, seconds):
    """把导出的 API 工作流里的 {{PROMPT}} 换成镜头提示词，并改随机种子和时长（秒）。返回新工作流和改了几处。"""
    wf = copy.deepcopy(wf)
    hits = {"prompt": 0, "seed": 0, "duration": 0}
    for node in wf.values():
        inputs = node.get("inputs", {})
        for key, value in list(inputs.items()):
            if isinstance(value, str) and "{{PROMPT}}" in value:
                inputs[key] = value.replace("{{PROMPT}}", prompt); hits["prompt"] += 1
            elif key in SEED_KEYS and isinstance(value, (int, float)) and not isinstance(value, bool):
                inputs[key] = seed; hits["seed"] += 1
            elif key in DURATION_KEYS and isinstance(value, (int, float)) and not isinstance(value, bool):
                inputs[key] = seconds; hits["duration"] += 1
    if hits["prompt"] == 0:
        raise ValueError("工作流里没有 {{PROMPT}}：在提示词框里填 {{PROMPT}} 后重新导出 API 格式")
    return wf, hits

def output_files(history_entry):
    """从 /history 的一条记录里找出所有输出文件（视频、图片、音频节点的写法不同，统一按带 filename 的字典找）。"""
    found = []
    def walk(value):
        if isinstance(value, dict):
            if "filename" in value and value.get("type", "output") == "output":
                found.append(value)
            for v in value.values():
                walk(v)
        elif isinstance(value, list):
            for v in value:
                walk(v)
    walk(history_entry.get("outputs", {}))
    return found

def seed_for(shot_id, take):
    return zlib.crc32(shot_id.encode()) % 100_000 * 100 + take

def queue(wf):
    body = json.dumps({"prompt": wf, "client_id": CLIENT}).encode()
    req = urllib.request.Request(f"{API}/prompt", data=body, headers={"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req))["prompt_id"]

def wait(prompt_id, timeout=3600):
    start = time.time()
    while time.time() - start < timeout:
        hist = json.load(urllib.request.urlopen(f"{API}/history/{prompt_id}"))
        if prompt_id in hist:
            entry = hist[prompt_id]
            status = entry.get("status", {})
            if status.get("status_str") == "error":
                raise RuntimeError(f"ComfyUI 报错：{status.get('messages')}")
            return entry, time.time() - start
        time.sleep(3)
    raise TimeoutError(prompt_id)

def fetch(file, dest):
    query = urllib.parse.urlencode({"filename": file["filename"], "subfolder": file.get("subfolder", ""), "type": file.get("type", "output")})
    with urllib.request.urlopen(f"{API}/view?{query}") as r, open(dest, "wb") as out:
        shutil.copyfileobj(r, out)

def run(shot_ids, takes=3, show=True):
    for shot in [s for s in SHOTS if s["id"] in shot_ids]:
        wf_path = WORK / "workflows" / f'{shot["group"]}.json'
        if not wf_path.exists():
            print(f'⚠️ 跳过 {shot["id"]}：还没有 {wf_path.name}（第 6 步）'); continue
        base = json.loads(wf_path.read_text())
        folder = WORK / "takes" / shot["id"]; folder.mkdir(parents=True, exist_ok=True)
        for t in range(1, takes + 1):
            seed = seed_for(shot["id"], t)
            wf, hits = patch_workflow(base, shot["prompt"], seed, shot["seconds"])
            if t == 1 and hits["duration"] == 0:
                print(f'ℹ️ 工作流里没找到时长参数，{shot["id"]} 用工作流里设的时长（应为 {shot["seconds"]} 秒）')
            entry, used = wait(queue(wf))
            for i, f in enumerate(output_files(entry)):
                ext = pathlib.Path(f["filename"]).suffix
                if ext.lower() not in (".mp4", ".webm", ".mov"):
                    continue
                dest = folder / f'take{t}-seed{seed}{"" if i == 0 else f"-{i}"}{ext}'
                fetch(f, dest)
                print(f'{shot["id"]} 第 {t} 段：{dest.name}，用时 {used / 60:.1f} 分钟')
                if show:
                    display(Video(str(dest), embed=True, width=480))
print("工具就绪")

## 8. 第一阶段：定下真的声音

先生成“名片”4 段（不给声音参考，由 H3 自己配音），挑声音最像真的那一段，把她说话的几秒截成 `shin-voice.wav`。之后的镜头都用这段当音色参考，声音才会前后一致。

In [ ]:
run(["card"], takes=4)

In [ ]:
CHOSEN = "take1-seed0.mp4"   # 改成你选中的那段文件名（在 takes/card/ 里）
START, END = 0.5, 8.5        # 真说话的那几秒，5–10 秒，不要带背景音乐
src = WORK / "takes" / "card" / CHOSEN
dst = WORK / "refs" / "shin-voice.wav"
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-ss", str(START), "-to", str(END), "-i", str(src), "-vn", "-ac", "1", "-ar", "32000", str(dst)], check=True)
copy_refs()
print("已保存", dst, "——现在回第 6 步做 shin.json、coding.json、notebook.json")

## 9. 第二阶段：生成其余镜头（每个镜头 3 段，挑最好的）

In [ ]:
run(["shadow", "notebook", "sin", "coding", "stars", "farewell"], takes=3)

## 10. 拼粗剪

在 `shin-intro/selected.json` 里写每个镜头选中的那段，例如：

```json
{"shadow": "take2-seed123.mp4", "card": "take1-seed456.mp4"}
```

没写的镜头跳过。粗剪只是先看整体，正式剪辑放回 Ryugi 分镜里做（加背景音乐、片尾网址）。

In [ ]:
selected = json.loads((WORK / "selected.json").read_text())
listing = BASE / "concat.txt"
with open(listing, "w") as f:
    for shot in SHOTS:
        if shot["id"] in selected:
            f.write(f"file '{WORK / 'takes' / shot['id'] / selected[shot['id']]}'\n")
out = WORK / "rough-cut.mp4"
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-f", "concat", "-safe", "0", "-i", str(listing),
                "-c:v", "libx264", "-crf", "18", "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "192k", str(out)], check=True)
display(Video(str(out), embed=True, width=640))

## 附：用上一段的最后一帧接下一段

两个镜头要连着（比如“小本子”翻到那页、接“本子里的 Sin”），可以截上一段最后一帧，在 ComfyUI 里当下一段的首帧参考。

In [ ]:
def last_frame(shot_id, take_file):
    src = WORK / "takes" / shot_id / take_file
    dst = COMFY / "input" / f"{shot_id}-last.png"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-sseof", "-0.1", "-i", str(src), "-frames:v", "1", str(dst)], check=True)
    return dst
# last_frame("notebook", "take2-seed123.mp4")

## 用完关机

In [ ]:
COMFY_PROC.terminate()
if "TUNNEL" in globals():
    TUNNEL.terminate()
print("ComfyUI 已关闭。Colab 记得在“代码执行程序”里断开并删除运行时，不然还在计费")